# 🎯 01-数据结构与算法 · 主题 2：双指针与滑动窗口

> 数组题两大高频套路。**双指针**把 $O(n^2)$ 暴力降为 $O(n)$，**滑动窗口**把「连续子区间」问题统一为 $O(n)$。
> 本 notebook 目标：**掌握三种指针形态 → 背熟窗口四要素 → 吃透 7 道高频题 → 通过 25 道自测**。

**前置**：主题 1（数组/哈希）。**运行环境**：Python 3.8+，无第三方依赖。

## §1 套路总览：三种形态 + 心智模型

### 三种形态

| 形态 | 指针移动 | 典型题 |
|------|---------|--------|
| **相向双指针** | 左指针向右、右指针向左 | 两数之和 II、盛水容器、接雨水、三数之和 |
| **同向快慢指针** | 快指针扫描、慢指针记录位置 | 移动零、去重、链表中点 |
| **滑动窗口** | 右指针扩展、左指针收缩，维护区间 | 无重复最长子串、最小覆盖子串 |

### 心智模型

- **相向**：两端夹逼，利用**单调性**缩小搜索空间——每次移动必然排除掉一部分不可能的解
- **同向**：快指针负责遍历，慢指针标记「下一个写入位置」，一趟原地完成
- **滑动窗口**：窗口 $[l, r)$ 是数组上一个连续区间；右扩找可行、左缩找最优，**每个元素最多进出窗口各一次**

### 什么时候用滑动窗口（判断口诀）

- ✅ 要求**连续子数组/子串** + 有**窗口条件的约束**（和、无重复、覆盖）
- ✅ 数组全为正数时「和」类问题；字符类问题天然适合
- ❌ 有负数时「和 == k」类问题**不能**用滑动窗口（窗口和单调性失效）→ 用前缀和（主题 1 的 560 题）

## §2 相向双指针：两数之和 II / 盛水容器

### 题 1：两数之和 II - 输入有序数组（LeetCode 167）

**题目**：有序数组，找两数之和等于 $target$。

**公式**：

$$a_{lo} + a_{hi} = target$$

**单调性推理**（为什么指针移动正确）：

若 $a_{lo} + a_{hi} < target$，说明和太小——因为 $a_{hi}$ 已是最大，只能增大 $a_{lo}$（$lo \leftarrow lo+1$）；
若和 $> target$，同理只能减小 $a_{hi}$（$hi \leftarrow hi-1$）。每次移动排除一行/一列候选，共 $O(n)$ 步。

### 题 2：盛最多水的容器（LeetCode 11）

**题目**：两条垂线之间能装多少水。

**公式**：

$$S(l, r) = \min(h_l, h_r) \times (r - l)$$

**关键推理**：固定 $l$ 时，$r$ 越远越好，但容量受**短板** $\min(h_l, h_r)$ 限制。若 $h_l < h_r$，任何其他右端点 $r'$ 与 $l$ 组成的容器，高度 $\le h_l$ 且宽度更小，必然更差——所以**移动短板一侧**，不会漏掉最优解。

**复杂度**：两者都是时间 $O(n)$，空间 $O(1)$。

In [ ]:
def two_sum_ii(numbers, target):
    # 有序数组：相向双指针，利用和的单调性
    lo, hi = 0, len(numbers) - 1
    while lo < hi:
        s = numbers[lo] + numbers[hi]
        if s == target:
            return [lo + 1, hi + 1]   # 本题下标从 1 开始
        elif s < target:
            lo += 1
        else:
            hi -= 1
    return []

def max_area(height):
    # S = min(h[l], h[r]) * (r - l)；永远移动短板一侧
    lo, hi = 0, len(height) - 1
    best = 0
    while lo < hi:
        h = min(height[lo], height[hi])
        best = max(best, h * (hi - lo))
        if height[lo] < height[hi]:
            lo += 1
        else:
            hi -= 1
    return best

print('two_sum_ii:', two_sum_ii([2, 7, 11, 15], 9))   # 期望 [1, 2]
print('two_sum_ii:', two_sum_ii([2, 3, 4], 6))        # 期望 [1, 3]
print('max_area:', max_area([1, 8, 6, 2, 5, 4, 8, 3, 7]))  # 期望 49
print('max_area:', max_area([1, 1]))                  # 期望 1

## §3 接雨水（LeetCode 42）：双指针公式推导

**题目**：每个位置能接多少雨水。

**核心公式**：位置 $i$ 的储水量由两侧最高柱子的短板决定：

$$\text{water}(i) = \max\left(0,\ \min(L_{max}(i),\ R_{max}(i)) - h_i\right)$$

其中 $L_{max}(i) = \max_{j \le i} h_j$，$R_{max}(i) = \max_{j \ge i} h_j$。

**朴素做法**：预处理 $L_{max}$、$R_{max}$ 两个数组，时间/空间 $O(n)$。

**双指针优化到 $O(1)$ 空间**：

- 左右指针夹逼，维护当前扫描过的 $\text{left\_max}$ 与 $\text{right\_max}$
- 关键推理：当 $h_{lo} < h_{hi}$ 时，**右侧存在高度 $\ge h_{hi}$ 的墙**，所以 $R_{max}(lo) \ge h_{hi} > h_{lo}$。此时 $\min(L_{max}, R_{max}) = L_{max}$ 完全由左侧已扫描部分决定——可以立刻结算 $lo$ 位置的水量
- 对称地，$h_{hi} \le h_{lo}$ 时结算 $hi$ 位置

**复杂度**：时间 $O(n)$，空间 $O(1)$。

> 👉 **指路**：接雨水的**单调栈版**见 04-栈与队列 §8「接雨水（LeetCode 42）：单调栈版」，可与本节双指针的短板推理对照理解。

In [ ]:
def trap(height):
    # water(i) = max(0, min(L_max, R_max) - h[i])
    lo, hi = 0, len(height) - 1
    left_max = right_max = 0
    total = 0
    while lo < hi:
        if height[lo] < height[hi]:
            # 右侧有足够高的墙兜底，L_max 已确定
            if height[lo] >= left_max:
                left_max = height[lo]      # 自己就是最高柱，存不住水
            else:
                total += left_max - height[lo]
            lo += 1
        else:
            if height[hi] >= right_max:
                right_max = height[hi]
            else:
                total += right_max - height[hi]
            hi -= 1
    return total

print('test1:', trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]))  # 期望 6
print('test2:', trap([4, 2, 0, 3, 2, 5]))                     # 期望 9
print('test3:', trap([1, 2, 3]))                              # 期望 0（单调，存不住水）

## §4 同向快慢指针：原地去重

### 题 3：删除有序数组中的重复项（LeetCode 26）

**题目**：原地删除重复元素，返回新长度。

**思路**：`slow` 指向「已去重区间」的末尾位置，`fast` 扫描。遇到新值就放到 `slow+1` 处并推进 `slow`。

**为什么成立**：数组有序，重复值必相邻；`nums[slow]` 永远是已扫描部分最后一个不同值。

**复杂度**：时间 $O(n)$，空间 $O(1)$。

> 💡 **同系列**：移除元素（27）、移动零（283，主题 1）、删除排序链表重复节点（83）。模板完全一样，只是「放回」条件不同。

In [ ]:
def remove_duplicates(nums):
    if not nums:
        return 0
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != nums[slow]:
            slow += 1
            nums[slow] = nums[fast]
    return slow + 1

nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(nums)
print('新长度 =', k, '，去重后前 k 项 =', nums[:k])  # 期望 [0,1,2,3,4]
print('空数组:', remove_duplicates([]))             # 期望 0

## §5 滑动窗口：万能模板（背这个）

### 四要素

1. **右扩**：`right` 每轮 +1，把新元素纳入窗口
2. **收缩**：while 不满足窗口约束时，移除 `left` 元素、`left` +1
3. **更新答案**：窗口合法时更新（通常取最大/最小）
4. **状态结构**：哈希表/计数器维护窗口内容

### 复杂度证明（为什么是 $O(n)$ 而不是 $O(n^2)$）

每个元素**恰好进窗口一次、出窗口一次**：

$$\sum_{right} O(1) + \sum_{left} O(1) = O(n)$$

左指针最多移动 $n$ 次（单调不减），整体线性。

### 两种「求答案」取向

- **最长类**（无重复最长子串）：窗口合法时**尝试扩展**，记录最大长度
- **最短类**（最小覆盖子串）：窗口合法时**尝试收缩**，记录最小长度

In [ ]:
# 滑动窗口通用骨架（四要素模板）
def sliding_window_template(data):
    n = len(data)
    left = 0
    window = {}                    # 1. 状态结构：哈希表记录窗口内容
    answer = None
    for right in range(n):
        add(data[right], window)   # 2. 右扩：纳入新元素
        while not valid(window):   # 3. 收缩：直到窗口重新合法
            remove(data[left], window)
            left += 1
        answer = update(answer, left, right)  # 4. 更新答案
    return answer

def add(x, w):  w[x] = w.get(x, 0) + 1
def remove(x, w):
    w[x] -= 1
    if w[x] == 0:
        del w[x]
def valid(w):   return True   # 由具体题目定义
def update(a, l, r): return a

# 演示：统计长度为 3 的固定窗口内最大元素和
def max_window_sum(nums, k):
    left = 0
    s = 0
    best = float('-inf')
    for right, x in enumerate(nums):
        s += x
        if right - left + 1 > k:      # 窗口超长：收缩
            s -= nums[left]
            left += 1
        if right - left + 1 == k:     # 窗口恰好为 k：更新答案
            best = max(best, s)
    return best

print('固定窗口 k=3 最大和:', max_window_sum([1, 3, -1, -3, 5, 3, 6, 7], 3))  # 期望 16

## §6 题 4：无重复字符的最长子串（LeetCode 3）

**题目**：最长不含重复字符的子串长度。

**思路**：窗口 $[l, r)$ 内字符不重复。遇到重复字符 $c$ 时，把 $l$ 跳到「上一次出现 $c$ 的位置 + 1」。

**公式**：

$$\text{answer} = \max_{r}(\, r - \text{last}[s_r] \, )$$

即每次右扩，答案就是「当前右端点 − 该字符上次出现位置」的最大值（隐式完成左边界跳跃，无需显式 while）。

**复杂度**：时间 $O(n)$，空间 $O(|\Sigma|)$（字符集大小）。

In [ ]:
def length_of_longest_substring(s):
    last = {}                      # 字符 -> 最近一次出现下标
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1    # 左边界跳到重复字符之后
        last[ch] = right
        best = max(best, right - left + 1)
    return best

print('test1:', length_of_longest_substring('abcabcbb'))  # 期望 3
print('test2:', length_of_longest_substring('bbbbb'))     # 期望 1
print('test3:', length_of_longest_substring('pwwkew'))    # 期望 3
print('test4:', length_of_longest_substring(''))          # 期望 0

## §7 题 5：最小覆盖子串（LeetCode 76，最短类窗口）

**题目**：在 $s$ 中找最短子串，包含 $t$ 的所有字符（含数量）。

**思路**：

1. `need` 哈希表记录 $t$ 中各字符还缺多少；`missing` 记录总缺量
2. 右扩纳入字符：若该字符仍缺则 `missing -= 1`
3. `missing == 0` 时窗口覆盖 $t$ → **尝试收缩**左边界求最短，同时恢复被移出字符的缺量

**复杂度**：$O(|s| + |t|)$ 时间，$O(|t|)$ 空间（只需记录 $t$ 的字符集）。

> ⚠️ **坑**：`need` 里出现负数（窗口内字符多于需要）是正常的——只有 `> 0` 才影响 `missing`。

In [ ]:
from collections import Counter

def min_window(s, t):
    if not s or not t:
        return ''
    need = Counter(t)
    missing = len(t)             # 还缺几个字符
    left = 0
    best = (0, float('inf'))     # (起点, 长度)
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:      # 窗口已覆盖 t，尝试收缩
            if right - left + 1 < best[1]:
                best = (left, right - left + 1)
            need[s[left]] += 1
            if need[s[left]] > 0:
                missing += 1
            left += 1
    return '' if best[1] == float('inf') else s[best[0]:best[0] + best[1]]

print('test1:', repr(min_window('ADOBECODEBANC', 'ABC')))  # 期望 'BANC'
print('test2:', repr(min_window('a', 'a')))                # 期望 'a'
print('test3:', repr(min_window('a', 'aa')))               # 期望 ''

## §8 题 6 / 题 7：最小子数组 + 字母异位词

### 题 6：长度最小的子数组（LeetCode 209）

**题目**：和 $\ge$ target 的最短连续子数组长度（全正数）。

**公式（窗口和）**：

$$\text{sum}(l, r) = \sum_{i=l}^{r} nums_i \ge target$$

**全正数 → 窗口和单调不减**，右扩使和增大、左缩使和减小，滑动窗口正确。

### 题 7：找到字符串中所有字母异位词（LeetCode 438）

**题目**：$s$ 中所有长度为 $|p|$、且是 $p$ 的字母异位词的起始下标。

**思路**：**固定窗口长度** $k = |p|$，维护窗口字符计数并与 $p$ 的计数比较。

**复杂度**：$O(n)$ 时间（计数比较是常数级 $|\Sigma|=26$），$O(1)$ 额外空间。

In [ ]:
def min_subarray_len(target, nums):
    left = 0
    s = 0
    best = float('inf')
    for right, x in enumerate(nums):
        s += x
        while s >= target:          # 窗口合法：收缩求最短
            best = min(best, right - left + 1)
            s -= nums[left]
            left += 1
    return 0 if best == float('inf') else best

print('min_subarray_len:', min_subarray_len(7, [2, 3, 1, 2, 4, 3]))   # 期望 2
print('min_subarray_len:', min_subarray_len(11, [1, 1, 1, 1]))        # 期望 0

from collections import Counter

def find_anagrams(s, p):
    target = Counter(p)
    window = Counter()
    res = []
    k = len(p)
    for i, ch in enumerate(s):
        window[ch] += 1
        if i >= k:                  # 固定窗口：移除窗口左侧字符
            c = s[i - k]
            window[c] -= 1
            if window[c] == 0:
                del window[c]
        if window == target:
            res.append(i - k + 1)
    return res

print('find_anagrams:', find_anagrams('cbaebabacd', 'abc'))   # 期望 [0, 6]
print('find_anagrams:', find_anagrams('abab', 'ab'))          # 期望 [0, 1, 2]

## §9 双指针进阶：判断子序列（392）与最长回文子串（5）

### 题 8：判断子序列（LeetCode 392）

**题目**：判断 $s$ 是否为 $t$ 的子序列（相对顺序一致即可，**不必连续**）。

**思路**：同向双指针——`i` 扫 $s$、`j` 扫 $t$：

- `s[i] == t[j]` 时 `i += 1`（匹配一个字符，指针前进）
- `j` 每轮都 `+1`（$t$ 一路扫到底）
- 结束时 `i == len(s)` 说明 $s$ 全部按序匹配 → True

**复杂度**：时间 $O(len(t))$，空间 $O(1)$。⚠️ 连续子串用滑动窗口，**不连续子序列**才是双指针/DP 的活（DP 版见主题 8）。

In [ ]:
def is_subsequence(s, t):
    i = j = 0
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1                 # 匹配成功才推进 s 的指针
        j += 1                     # t 的指针始终右移
    return i == len(s)             # s 全部按序匹配完

assert is_subsequence('abc', 'ahbgdc') is True
assert is_subsequence('axc', 'ahbgdc') is False
assert is_subsequence('', 'ahbgdc') is True       # 空串是任何串的子序列
assert is_subsequence('abc', '') is False
print('392 is_subsequence OK')

### 题 9：最长回文子串（LeetCode 5）：中心扩展

**题目**：回文串正读反读相同；在 $s$ 中找最长的回文子串。

**思路**：中心扩展——每个位置都能作为回文中心，向两边同步扩展直到字符不等：

- 奇数长度回文的中心是一个字符：`(i, i)`
- 偶数长度回文的中心是两个字符的间隙：`(i, i + 1)`

对每个中心向两侧扩展并记录最长。总中心 $2n-1$ 个、每次扩展 $O(n)$ → **$O(n^2)$**。

> 💡 Manacher 算法可优化到 $O(n)$，面试写出中心扩展即可；注意区分：回文**子串**（连续）用中心扩展，回文**子序列**用 DP。

In [ ]:
def longest_palindrome(s):
    def expand(l, r):            # 从中心 (l, r) 向两边扩展
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1
            r += 1
        return s[l + 1:r]        # 回文区间 [l+1, r)

    best = ''
    for i in range(len(s)):
        for l, r in ((i, i), (i, i + 1)):   # 奇数中心 / 偶数中心
            cand = expand(l, r)
            if len(cand) > len(best):
                best = cand
    return best

assert longest_palindrome('babad') in ('bab', 'aba')
assert longest_palindrome('cbbd') == 'bb'
assert longest_palindrome('a') == 'a'
assert longest_palindrome('') == ''
print('5 longest_palindrome OK')

## §10 复杂度总表（背熟）

| 套路 | 时间复杂度 | 空间复杂度 | 说明 |
|------|-----------|-----------|------|
| 相向双指针（有序两数之和） | $O(n)$ | $O(1)$ | 单调性剪枝 |
| 相向双指针（盛水/接雨水） | $O(n)$ | $O(1)$ | 短板推理 |
| 同向快慢（去重/移动零） | $O(n)$ | $O(1)$ | 一趟原地 |
| 滑动窗口（最长类） | $O(n)$ | $O(\|\Sigma\|)$ | 右扩为主 |
| 滑动窗口（最短类） | $O(n)$ | $O(\|\Sigma\|)$ | 左缩为主 |
| 固定窗口（异位词） | $O(n)$ | $O(1)$ | 计数比较 |

**为什么都是 $O(n)$**：双指针/窗口的左右指针**各自单调移动**，总步数 $O(n)$，不存在回溯。

## §11 常见 bug 与边界（面试踩坑区）

- **while 忘写 `left < right`**：相向指针会交叉，导致重复计算/死循环
- **窗口收缩条件写反**：最短类用 while 收缩，最长类通常不用 while（跳跃式更新 left）
- **固定窗口的删除时机**：先判断 `i >= k` 再移除 `s[i-k]`，顺序错了会漏窗口
- **负数不能用滑动窗口求和**：窗口和没有单调性（主题 1 的 560 题用前缀和）
- **字符计数为 0 要 del**：`Counter` 比较时残留 0 计数字符会导致 `!=` 误判
- **`float('inf')` 初始值**：最短类结果判空用 `inf` 哨兵，别用 `-1`
- **移动端点而非比较端点**：接雨水判断 `height[lo] < height[hi]` 决定结算哪一侧，别写成比较后忘记移动
- **子串 vs 子序列**：滑动窗口只解决**连续**子串问题，子序列要用 DP（主题 8）

## §12 面试追问与扩展（加分项）

### 12.1 为什么相向双指针不漏解

每次移动排除的是一个「不可能成为最优」的子区间集合——靠的是问题内在的**单调性**（和的大小、短板高度）。能现场讲清这一点，比背代码加分得多。

### 12.2 滑动窗口 vs 前缀和 vs DP 的选择

- 窗口和问题 + 全正数 → 滑动窗口
- 子数组和 == k（含负数）→ 前缀和 + 哈希（560）
- 最大子数组和（可任意跨越）→ 一维 DP / Kadane（53）

### 12.3 变体

- 至多 k 个不同字符的最长子串（340）
- 替换后最长重复字符（424）——窗口内最多允许 k 个「其他字符」
- 滑动窗口 + 单调队列：滑动窗口最大值（239，主题 4）

## §13 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 无重复字符的最长子串（LeetCode 3）
- [ ] 2. 长度最小的子数组（209）
- [ ] 3. 删除有序数组中的重复项（26）
- [ ] 4. 移除元素（27）
- [ ] 5. 两数之和 II（167）
- [ ] 6. 验证回文串（125）
- [ ] 7. 反转字符串（344）
- [ ] 8. 移动零（283）
- [ ] 9. 反转链表（206，链表主题）
- [ ] 10. 链表的中间结点（876）

### L2 进阶（10 题）

- [ ] 11. 最小覆盖子串（76）
- [ ] 12. 盛最多水的容器（11）
- [ ] 13. 接雨水（42）
- [ ] 14. 三数之和（15）
- [ ] 15. 四数之和（18）
- [ ] 16. 找到字符串中所有字母异位词（438）
- [ ] 17. 最长回文子串（5，中心扩展）
- [ ] 18. 字符串的排列（567）
- [ ] 19. 合并两个有序数组（88）
- [ ] 20. 有序数组的平方（977）

### L3 挑战（5 题）

- [ ] 21. 至多包含 K 个不同字符的最长子串（340）
- [ ] 22. 替换后的最长重复字符（424）
- [ ] 23. 滑动窗口最大值（239，单调队列）
- [ ] 24. 子数组最大平均数 II（644，浮点二分 + 窗口）
- [ ] 25. 环形子数组的最大和（918）

### 自测要点

- 背写滑动窗口四要素模板（右扩/收缩/更新/状态结构）
- 讲清为什么每个元素进出窗口各一次 → 复杂度 $O(n)$
- 说出「哪些场景不能用滑动窗口」（负数、子序列）

## 附录：参考与下节预告

- 《算法导论》：双指针问题散见于排序与选择、贪心章节
- LeetCode 分类题单：双指针 / 滑动窗口

### 下节预告：主题 3 — 链表

- 反转链表（迭代/递归）、环形链表（Floyd 判圈推导）、LRU 缓存、K 个一组翻转、回文链表

> 💡 本主题验收：`高频题单.md`「双指针/滑动窗口」一节全部打勾，能 10 分钟盲写无重复最长子串 + 最小覆盖子串。